In [0]:
aapl_data = spark.sql('select * from default.aapl_data')
display(aapl_data)

In [0]:
def diff_high_low(high, low):
    high_num = float(high.lstrip('$'))
    low_num = float(low.lstrip('$'))
    return (f'${round(high_num - low_num, 2)}')

In [0]:
from pyspark.sql.types import *
compute_diff_udf = udf(diff_high_low, StringType())

aapl_data_new = aapl_data.withColumn('diff_high_low', compute_diff_udf(aapl_data['High'], aapl_data['Low']))\
    .select('date', 'volume', 'open', 'high', 'low', 'diff_high_low')\
        .display()

In [0]:
aapl_new_2 = aapl_data.select('date', 'volume', 'open', 'high', 'low',
                              compute_diff_udf('high', 'low').alias('diff_high_low'))
aapl_new_2.display()
#can use udf with select statment as well

In [0]:
from typing import Optional

@udf(returnType= FloatType())
def change_percent(high, low) -> Optional[float]:
    if low:
        high_num = float(high.lstrip('$'))
        low_num = float(low.lstrip('$'))
        return ((high_num - low_num) / low_num) * 100
    else:
        return None
#here didnt need to explicitly convert the function into udf first

In [0]:
aapl_new_3 = aapl_data.select('date', 'volume', 'open', 'high', 'low',
                              change_percent('high', 'low').alias('change_percent'))
aapl_new_3.display()

In [0]:
spark.udf.register('change_percent', change_percent)
#need to register the udf first to use inside sql query
aapl_data.createOrReplaceTempView('aapl_data_temp')
spark.sql("""select date, volume, open, high, low, change_percent(high, low) as `Change Percent`
           from aapl_data_temp""").display()

In [0]:
import pandas as pd
from pyspark.sql.functions import pandas_udf, col, PandasUDFType
from pyspark.sql.types import IntegerType

from typing import Iterator, Tuple

In [0]:
def year(date: pd.Series) -> pd.Series:
    return(pd.to_datetime(date).dt.year)
#vectorized udf
year_pandas = pandas_udf(year, returnType=IntegerType())
aapl_data.withColumn('Year', year_pandas(col('date')))\
    .select('date', 'volume', 'open', 'Year').display()

In [0]:
def stop_loss_percent():
    return 0.07

In [0]:
@pandas_udf("string")
def compute_stop_loss(iterator: Iterator[pd.Series]) -> Iterator[pd.Series]:
    stoplosspercent = stop_loss_percent()
    for open_price in iterator:
        result = round(open_price.str.lstrip('$').astype(float) * (1 - stoplosspercent), 2)
        yield '$' + result.astype(str)


In [0]:
aapl_data.select('date', 'Volume', 'Open', 'high', 'low',
                 compute_stop_loss(col('open')).alias('stoploss')).display()
#lstrip not work on iterator, so converted to series

In [0]:
#Can use iterator of multiple series, like by using  series in a form of tuple and iterator for that tuple
# def open_and_close_price(iterator: Iterator[Tuple[pd.Series, pd.Series]]) -> Iterator[pd.Series]:
#     for open_price, close_price in iterator:
#         yield open_price + close_price

In [0]:
#iterator to scaler
@pandas_udf('float')
def average(values: pd.Series) -> float:
    return values.mean()